# Virtual Mouse / Air Canvas - Phase 4 (final)

## Tuned to your hand: clean click + two-finger drag

Thresholds set from your measured values (index-thumb ~0.10 when touching,
middle-thumb stays ~0.45+).

| Action | Gesture |
|--------|---------|
| **Move** | point with index finger |
| **Click** | index + thumb touch (middle finger OUT) - cursor freezes |
| **Drag** | pinch index+thumb with the **middle finger curled DOWN**, move, release |
| **Scroll** | index + middle both up (not pinched), move hand up/down |

Control gate: **open palm** = ON, **fist** = OFF.

---
### How click and drag stay separate
- **Click** = index-thumb pinch with the **middle finger UP** (extended).
- **Drag** = index-thumb pinch with the **middle finger curled DOWN**.
We use the middle finger's up/down state (reliable) rather than its distance to
the thumb (which barely changes on your hand), so click and drag never blur.

### Safety
Open palm = ON, fist = OFF, **`q`** to quit, mouse to a corner = abort.
**Restart the kernel, then run all cells top to bottom.**

## 1. Imports

In [1]:
import time
import math
import platform
import threading
from collections import deque

import cv2
import numpy as np
import mediapipe as mp
import pyautogui

assert hasattr(mp.solutions, "hands"), "Need mediapipe==0.10.21"
pyautogui.FAILSAFE = True
pyautogui.PAUSE = 0
SCREEN_W, SCREEN_H = pyautogui.size()
print("Imports OK. Screen:", SCREEN_W, "x", SCREEN_H)

Imports OK. Screen: 1920 x 1080


## 2. Threaded camera

In [2]:
class ThreadedCamera:
    def __init__(self, src=0, width=640, height=480):
        if platform.system() == "Windows":
            self.cap = cv2.VideoCapture(src, cv2.CAP_DSHOW)
        else:
            self.cap = cv2.VideoCapture(src)
        self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, width)
        self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, height)
        self.buffer = deque(maxlen=1)
        self.running = False
        self.lock = threading.Lock()
        self.thread = None

    def opened(self):
        return self.cap.isOpened()

    def start(self):
        self.running = True
        self.thread = threading.Thread(target=self._loop, daemon=True)
        self.thread.start()
        return self

    def _loop(self):
        while self.running:
            ok, frame = self.cap.read()
            if ok:
                with self.lock:
                    self.buffer.append(frame)

    def read(self):
        with self.lock:
            return self.buffer[-1] if self.buffer else None

    def stop(self):
        self.running = False
        if self.thread is not None and self.thread.is_alive():
            self.thread.join(timeout=1)
        self.cap.release()

## 3. Hand tracker

In [3]:
WRIST = 0
THUMB_TIP, INDEX_TIP, MIDDLE_TIP, RING_TIP, PINKY_TIP = 4, 8, 12, 16, 20
INDEX_MCP = 5
TIPS = [4, 8, 12, 16, 20]
PIPS = [3, 6, 10, 14, 18]


class HandTracker:
    def __init__(self, max_hands=1, detection_conf=0.7, tracking_conf=0.5):
        self.mp_hands = mp.solutions.hands
        self.mp_draw = mp.solutions.drawing_utils
        self.mp_styles = mp.solutions.drawing_styles
        self.hands = self.mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=max_hands,
            min_detection_confidence=detection_conf,
            min_tracking_confidence=tracking_conf,
        )

    def process(self, frame_bgr):
        rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        results = self.hands.process(rgb)
        rgb.flags.writeable = True
        return results

    def draw(self, frame_bgr, results):
        if results.multi_hand_landmarks:
            for lm in results.multi_hand_landmarks:
                self.mp_draw.draw_landmarks(
                    frame_bgr, lm, self.mp_hands.HAND_CONNECTIONS,
                    self.mp_styles.get_default_hand_landmarks_style(),
                    self.mp_styles.get_default_hand_connections_style(),
                )

    @staticmethod
    def landmark_pixel(hand_landmarks, index, frame_shape):
        h, w = frame_shape[:2]
        lm = hand_landmarks.landmark[index]
        return int(lm.x * w), int(lm.y * h)

    @staticmethod
    def fingers_up(hand_landmarks):
        lm = hand_landmarks.landmark
        fingers = [1 if lm[THUMB_TIP].x < lm[THUMB_TIP - 1].x else 0]
        for tip, pip in zip(TIPS[1:], PIPS[1:]):
            fingers.append(1 if lm[tip].y < lm[pip].y else 0)
        return fingers

    @staticmethod
    def _hand_size(lm):
        return math.hypot(lm[INDEX_MCP].x - lm[WRIST].x,
                          lm[INDEX_MCP].y - lm[WRIST].y) + 1e-6

    @classmethod
    def pinch_ratio(cls, hand_landmarks, a=INDEX_TIP, b=THUMB_TIP):
        lm = hand_landmarks.landmark
        d = math.hypot(lm[a].x - lm[b].x, lm[a].y - lm[b].y)
        return d / cls._hand_size(lm)

    def close(self):
        self.hands.close()

## 4. EMA smoothing

In [4]:
class EMAFilter:
    def __init__(self, alpha=0.3):
        self.alpha = alpha
        self.x = None
        self.y = None

    def filter(self, x, y):
        if self.x is None:
            self.x, self.y = x, y
        else:
            self.x = self.alpha * x + (1 - self.alpha) * self.x
            self.y = self.alpha * y + (1 - self.alpha) * self.y
        return self.x, self.y

    def reset(self):
        self.x = self.y = None

## 5. Debounced edge detector

In [5]:
class Debounced:
    def __init__(self, frames=3):
        self.frames = frames
        self.state = False
        self.cand = False
        self.count = 0

    def update(self, raw):
        if raw != self.cand:
            self.cand = raw
            self.count = 1
        else:
            self.count += 1
        rising = False
        if self.count >= self.frames and self.state != self.cand:
            rising = self.cand and not self.state
            self.state = self.cand
        return self.state, rising

    def reset(self):
        self.state = self.cand = False
        self.count = 0

## 6. Tuning knobs (set from YOUR measured values)

- `CLICK_ON = 0.18` - your index-thumb touches at ~0.10, so 0.18 fires only
  when genuinely close (not on approach, which was the early-click bug).
- **Drag** = index-thumb pinch while the **middle finger is curled down**.
  This uses the reliable middle finger up/down state, not the middle-thumb
  distance (which barely changes on your hand).

In [6]:
ALPHA = 0.3
ACTIVE_LEFT, ACTIVE_RIGHT = 0.30, 0.70
ACTIVE_TOP, ACTIVE_BOTTOM = 0.25, 0.65
EDGE_MARGIN = 3

CLICK_ON = 0.18          # index-thumb below this = pinch (your touch ~0.10)
DEBOUNCE_FRAMES = 3
SCROLL_SENSITIVITY = 12
SCROLL_DEADZONE = 0.004

def map_to_screen(nx, ny):
    sx = np.interp(nx, (ACTIVE_LEFT, ACTIVE_RIGHT), (EDGE_MARGIN, SCREEN_W - EDGE_MARGIN))
    sy = np.interp(ny, (ACTIVE_TOP, ACTIVE_BOTTOM), (EDGE_MARGIN, SCREEN_H - EDGE_MARGIN))
    return sx, sy

## 7. Run

- **Open palm** -> ON, **fist** -> OFF
- **Point** (index only) -> move
- **Index+thumb pinch** (middle out) -> CLICK (cursor freezes)
- **Index+middle BOTH pinch thumb** -> DRAG (move, release to drop)
- **Index+middle up** (not pinched), move up/down -> SCROLL
- **`q`** to quit | mouse to a corner -> abort

In [7]:
camera = ThreadedCamera(src=0)
if not camera.opened():
    print("ERROR: could not open camera. Close other camera apps / try src=1.")
else:
    camera.start()
    time.sleep(1.0)
    tracker = HandTracker(max_hands=1)
    ema = EMAFilter(alpha=ALPHA)
    click_db = Debounced(DEBOUNCE_FRAMES)
    drag_db = Debounced(DEBOUNCE_FRAMES)

    control_enabled = False
    dragging = False
    prev = time.time()
    prev_scroll_y = None
    print("Running. palm=ON fist=OFF | index+thumb=click | index+middle pinch=drag | two up=scroll | q=quit")

    try:
        while True:
            frame = camera.read()
            if frame is None:
                continue
            frame = cv2.flip(frame, 1)
            h, w = frame.shape[:2]
            now = time.time()
            results = tracker.process(frame)
            tracker.draw(frame, results)

            x1, y1 = int(ACTIVE_LEFT * w), int(ACTIVE_TOP * h)
            x2, y2 = int(ACTIVE_RIGHT * w), int(ACTIVE_BOTTOM * h)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 200, 0), 2)

            action_text = ""
            if results.multi_hand_landmarks:
                hand = results.multi_hand_landmarks[0]
                fingers = tracker.fingers_up(hand)
                total = sum(fingers)

                if total == 5:
                    control_enabled = True
                elif total == 0:
                    control_enabled = False
                    ema.reset(); click_db.reset(); drag_db.reset()
                    if dragging:
                        pyautogui.mouseUp(); dragging = False
                    prev_scroll_y = None

                if control_enabled:
                    two_up = fingers[1] == 1 and fingers[2] == 1 and total == 2

                    click_ratio = tracker.pinch_ratio(hand, INDEX_TIP, THUMB_TIP)
                    index_pinched = click_ratio < CLICK_ON
                    middle_up = fingers[2] == 1
                    # CLICK = index-thumb pinch with middle finger UP (extended)
                    click_raw = index_pinched and middle_up
                    # DRAG  = index-thumb pinch with middle finger CURLED down
                    drag_raw = index_pinched and not middle_up

                    drag_state, _ = drag_db.update(drag_raw)
                    click_state, click_rising = click_db.update(click_raw)

                    if two_up and not index_pinched:
                        cy = hand.landmark[INDEX_TIP].y
                        if prev_scroll_y is not None:
                            dy = prev_scroll_y - cy
                            if abs(dy) > SCROLL_DEADZONE:
                                pyautogui.scroll(int(dy * SCROLL_SENSITIVITY * 100))
                        prev_scroll_y = cy
                        action_text = "SCROLL"
                    else:
                        prev_scroll_y = None

                        if drag_state:
                            if not dragging:
                                pyautogui.mouseDown(); dragging = True
                            tip = hand.landmark[INDEX_TIP]
                            sx, sy = map_to_screen(tip.x, tip.y)
                            sx, sy = ema.filter(sx, sy)
                            pyautogui.moveTo(float(np.clip(sx, EDGE_MARGIN, SCREEN_W - EDGE_MARGIN)),
                                             float(np.clip(sy, EDGE_MARGIN, SCREEN_H - EDGE_MARGIN)))
                            action_text = "DRAGGING"
                        else:
                            if dragging:
                                pyautogui.mouseUp(); dragging = False

                            if click_state:
                                # freeze cursor during click
                                if click_rising:
                                    pyautogui.click()
                                    action_text = "CLICK"
                                else:
                                    action_text = "PINCH (hold)"
                            else:
                                tip = hand.landmark[INDEX_TIP]
                                sx, sy = map_to_screen(tip.x, tip.y)
                                sx, sy = ema.filter(sx, sy)
                                pyautogui.moveTo(float(np.clip(sx, EDGE_MARGIN, SCREEN_W - EDGE_MARGIN)),
                                                 float(np.clip(sy, EDGE_MARGIN, SCREEN_H - EDGE_MARGIN)))
                                action_text = "MOVE"

                    cx, cy = tracker.landmark_pixel(hand, INDEX_TIP, frame.shape)
                    if drag_state:
                        dot = (255, 0, 255)      # magenta = drag
                    elif click_state:
                        dot = (0, 0, 255)        # red = click pinch
                    else:
                        dot = (0, 255, 0)        # green = open
                    cv2.circle(frame, (cx, cy), 10, dot, cv2.FILLED)

            label = "CONTROL: ON" if control_enabled else "CONTROL: OFF"
            color = (0, 200, 0) if control_enabled else (0, 0, 255)
            cv2.putText(frame, label, (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
            if action_text:
                cv2.putText(frame, action_text, (10, 90),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
            fps = 1.0 / (now - prev) if now != prev else 0.0
            prev = now
            cv2.putText(frame, f"FPS: {int(fps)}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

            cv2.imshow("Phase 4 - Virtual Mouse", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    finally:
        if dragging:
            try: pyautogui.mouseUp()
            except Exception: pass
        camera.stop()
        tracker.close()
        cv2.destroyAllWindows()
        print("Stopped cleanly.")

Running. palm=ON fist=OFF | index+thumb=click | index+middle pinch=drag | two up=scroll | q=quit
Stopped cleanly.


## 8. Cleanup (run if the window hangs)

In [8]:
try:
    camera.stop()
except Exception:
    pass
cv2.destroyAllWindows()
for _ in range(4):
    cv2.waitKey(1)
print("cleaned up")

cleaned up


## Tuning tips (dot colours: green=open, red=click, magenta=drag)
- **Click still early?** Lower `CLICK_ON` toward 0.14.
- **Click not firing?** Raise `CLICK_ON` toward 0.22.
- **Drag won't start?** Make sure your middle finger curls DOWN while you
  pinch index+thumb. Middle UP = click, middle DOWN = drag.
- **Click turns into drag?** Keep your middle finger extended UP during a click.
- Watch the dot: it should be RED for a clean click, MAGENTA only when you
  deliberately fold both fingers in for a drag.